<a href="https://colab.research.google.com/github/GabrielMV07/EBAC/blob/main/Atv_PLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import plotly.figure_factory as ff

# utilizei a IA do google, o Gemini, para me ajudar a montar o arquivo em .csv, para usar como dataframe na atividade, pedi a ele para escrever 300 manchetes de
# notícias do Brasi e do mundo nos anos de 2025 e 2026, com 5 temas diferentes, depois colei linha por linha numa planilha com os temas correspondentes e salvei o
# arquivo em .csv

from google.colab import drive

drive.mount('/content/drive' , force_remount=True)

cm = '/content/drive/MyDrive/Curso/Notícias.csv'

df = pd.read_csv(cm, sep=',')

pl = ''

df.head()

Mounted at /content/drive


,ID,Manchetes,Temas
0,1,"Fitch eleva previsão do PIB do Brasil para 2,1...",Economia
1,2,Lula visita fábrica de fertilizantes da Petrob...,Política
2,3,Ministério da Saúde emite alerta e investiga c...,Saúde
3,4,Brasil supera Egito em último compromisso amis...,Esportes
4,5,Hospitais brasileiros adotam ecossistemas inte...,Saúde


In [ ]:
print(df.info())

print(pl)

# verificando a distribuição dos temas

print(df['Temas'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         300 non-null    int64 
 1   Manchetes  300 non-null    object
 2   Temas      300 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.2+ KB
None

Temas
Economia      62
Política      61
Saúde         60
Esportes      59
Tecnologia    58
Name: count, dtype: int64


In [ ]:
 # conferindo a media de aparição das palavras mais frequentes

tfidf1 = TfidfVectorizer(max_features=1000)

manchetes_vetorizadas = tfidf1.fit_transform(df['Manchetes'])

palavras = tfidf1.get_feature_names_out()

media = manchetes_vetorizadas.toarray().mean(axis=0)

df_media = pd.DataFrame({'Palavras': palavras, 'Média': media})

df_media = df_media.sort_values(by='Média', ascending=False)

df_media.head(20)

,Palavras,Média
252,de,0.092949
802,para,0.046815
288,do,0.045730
329,em,0.042462
752,no,0.032746
249,da,0.031635
192,com,0.027397
739,na,0.023773
34,anuncia,0.019411
741,nacional,0.016247


In [ ]:
# removendo palavras

remover = [ 'de' , 'para' , 'do' , 'dos' , 'em' , 'no' , 'da' , 'das' , 'com' , 'na' ,'ao' , 'que' ,'por']

tfidf2 = TfidfVectorizer(max_features=1000, stop_words=remover)

manchetes_vetorizadas2 = tfidf2.fit_transform(df['Manchetes'])

palavras2 = tfidf2.get_feature_names_out()

media2 = manchetes_vetorizadas2.toarray().mean(axis=0)

df_media2 = pd.DataFrame({'Palavras': palavras2, 'Média': media2})

df_media2 = df_media2.sort_values(by='Média', ascending=False)

# verificação

df_media2.head(20)

,Palavras,Média
32,anuncia,0.020380
746,nova,0.016755
168,contra,0.016464
731,nacional,0.016458
908,saúde,0.015358
38,apresenta,0.014290
39,aprova,0.014213
618,lança,0.013915
749,novos,0.013886
935,são,0.013567


In [ ]:
# transformando a coluna "Temas"

le = LabelEncoder()

df['Temas'] = le.fit_transform(df['Temas'])

# fazendo a separacao das variaveis

x = tfidf2.fit_transform(df['Manchetes'])

y = df['Temas']

x_treino, x_teste, y_treino, y_teste = train_test_split(x , y , test_size=0.2 , random_state=42)

modelo = LogisticRegression()

modelo.fit(x_treino, y_treino)

# conferindo resultados

previsoes = modelo.predict(x_teste)

print(classification_report(y_teste, previsoes))

matriz_confusao = ff.create_annotated_heatmap(confusion_matrix(y_teste, previsoes), x = ['Economia' , 'Esportes' , 'Política' , 'Saúde' , 'Tecnologia'] ,
                                              y = ['Economia' , 'Esportes' , 'Política' , 'Saúde' , 'Tecnologia'] , colorscale='blues')
matriz_confusao.update_layout(title='Matriz de Confusão' , xaxis_title='Valores Preditos' , yaxis_title='Valores Reais')
matriz_confusao.show()

              precision    recall  f1-score   support

           0       0.80      1.00      0.89        12
           1       0.92      0.92      0.92        13
           2       1.00      0.50      0.67        12
           3       0.67      0.83      0.74        12
           4       0.73      0.73      0.73        11

    accuracy                           0.80        60
   macro avg       0.82      0.80      0.79        60
weighted avg       0.83      0.80      0.79        60



**Boas práticas usadas**

1 - verificação de cada etapa

eu considero importante sempre verificar cada processo de cada etapa, desde a manipulação dos dados até a construção do código do modelo, ajuda a minimizar e corrigir erros, como nessa atividade eu mesmo montei a base de dados, não foi necessário manipular os dados, mas ainda sim fiz a verificação deles no começo

---
2 - remoção de artigos

a remoção de palavras de artigo melhora a compreensão do modelo, já que são palavras "sem significado" quando estão sozinhas e também aparecem muitas vezes em diversas manchetes diferentes

---
3 - avaliação

além de usar o "classification_report" para avaliar o modelo, gosto de fazer um mapa de calor das previsões, assim pode se visualizar com mais clareza as falhas do modelo ajudando a entender melhor onde ele está errando

**Breve reflexão sobre LGPD**

um modelo de categorização dos temas de notícias seria de certa forma sensível a LGPD, já que as manchetes usadas foram escritas por diversos jornalistas, em períodos e lugares diferentes, mas considerando que esses profissionais já trabalham com ética para evitar problemas a eles mesmos, em tese já garante que o modelo não utilize dados pessoais sensíveis dos indivíduos citados em questão, pois os próprios jornalista e autores, precisam tomar esse cuidado até mesmo além da manchete no decorrer da notícia. O de fato que poderia ser preocupante com um modelo desse tipo seria as associações equivocadas que ele poderia fazer, como por exemplo ligar o nome de pessoas (públicas ou não) a contextos dos quais elas não pertecem e podem ofende-las, como por exemplo se além dos 5 temas já do escolhidos, também houvesse o tema de "crimes" e o modelo categorizasse manchetes que citam os nomes de pessoa públicas ou organizações como notítcias sobre crimes cometidos, onde por mais que o contexto da matéria explique claramente do que se trata, a repercussão negativa que teria só por essa associação negativa já causaria problemas e constrangimento, por isso é bom usar temas abrangentes, que não façam associações ofensivas graves, claro que o tema "política" se trata de um tópico extremamente sensível ainda para pessoas públicas que não se envolvem com o tema por opção, sendo melhor uma análise manual desse tema em questão